In [37]:
import langsmith
print(langsmith.__version__)

0.4.29


In [38]:
import os

In [ ]:
# keys

In [40]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
output_parser = StrOutputParser()

In [41]:
chain = llm | output_parser
response = chain.invoke("Tell me a short joke about programming")

print(response)

Why do programmers prefer dark mode?

Because light attracts bugs!


In [42]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Overall rating out of 5")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")

review_text = """
Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
colors pop like crazy. Camera's insane too, especially at night - my Insta game's never been
stronger. Battery life's solid, lasts me all day no problem.
Not gonna lie though, it's pretty pricey. And what's with ditching the charger? C'mon Samsung.
Also, still getting used to the new button layout, keep hitting Bixby by mistake.
Overall, I'd say it's a solid 4 out of 5.
"""

structured_llm = llm.with_structured_output(MobileReview)
output = structured_llm.invoke(review_text)
print(output)

phone_model='Galaxy S21' rating=4.0 pros=['Gorgeous screen', 'Insane camera (especially at night)', 'Solid battery life'] cons=['Pricey', 'No charger included', 'New button layout (accidental Bixby presses)'] summary="Excellent phone with a gorgeous screen, great camera, and solid battery, but it's pricey and lacks a charger."


In [43]:
output.cons

['Pricey',
 'No charger included',
 'New button layout (accidental Bixby presses)']

In [44]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")
prompt.invoke({"topic":"programming"})


ChatPromptValue(messages=[HumanMessage(content='Tell me a short joke about programming', additional_kwargs={}, response_metadata={})])

In [45]:
chain = prompt | llm | output_parser
chain.invoke({"topic": "racing"})

"Why did the race car break up with the electric car?\n\nIt just couldn't **charge** ahead!"

In [46]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define Prompt
prompt  = ChatPromptTemplate.from_template("Give detailed information on {topic}")

# Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Define output parser
output_parser = StrOutputParser()

# Compose the chain
chain = prompt | llm | output_parser

# Use the chain
result = chain.invoke({"topic":"nuclear physics"})
print(result)


Nuclear physics is a fascinating and fundamental branch of physics that delves into the heart of matter: the atomic nucleus. It explores the structure, properties, and behavior of atomic nuclei, as well as the forces that bind them together and the processes by which they transform.

Here's a detailed look into nuclear physics:

---

## 1. Introduction to Nuclear Physics

**Definition:** Nuclear physics is the field of physics that studies the constituents and interactions of atomic nuclei. It also studies other forms of nuclear matter.

**Distinction from Atomic Physics:** While atomic physics focuses on the atom as a whole, particularly the electron shells and their interactions with the nucleus, nuclear physics zeroes in on the nucleus itself. The energy scales involved in nuclear processes are typically millions of times larger than those in atomic processes.

**Key Forces:** Nuclear physics is primarily concerned with two of the four fundamental forces of nature:
*   **Strong Nucl

In [47]:
template = ChatPromptTemplate([
    ("system", "You are a funny assistant that knows when to give very precise and specific output and when to give a detailed output with a short summary of keypoints"),
    ("human", "Tell me about {topic}")
])

prompt_value = template.invoke(
    {
        "topic":"programming"
    }
)

prompt_value

ChatPromptValue(messages=[SystemMessage(content='You are a funny assistant that knows when to give very precise and specific output and when to give a detailed output with a short summary of keypoints', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about programming', additional_kwargs={}, response_metadata={})])

In [48]:
llm.invoke(prompt_value)

AIMessage(content='Alright, buckle up, buttercup! You\'ve asked about programming, and let me tell you, it\'s a wild ride through the land of ones and zeros, where logic is king and semicolons are the tiny, often forgotten, rulers of your sanity.\n\n---\n\n### What in the Binary Blazes is Programming?\n\nImagine you have a super-fast, incredibly powerful, but utterly clueless assistant. This assistant can do *anything* you tell it, but only if you tell it *exactly* what to do, step-by-step, with no ambiguity whatsoever. That assistant is your computer, and giving it those super-precise instructions is **programming**.\n\nIn essence, programming is the art of writing a set of instructions (called "code") that a computer can understand and execute to perform a specific task or solve a problem. It\'s like writing a recipe, but instead of "add a pinch of salt," you\'re saying "take the value from memory address 0xAF, add 1 to it, and store the result back in memory address 0xAF." (Don\'t w

In [49]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.documents import Document
import os

def load_documents(folder_path: str) -> List[Document]:
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif filename.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents

folder_path = "docs"
documents = load_documents(folder_path)
print(f"Loaded {len(documents)} documents from the folder.")


Loaded 1 documents from the folder.


In [50]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks.")

Split the documents into 3 chunks.


In [51]:
documents[0]

Document(metadata={'source': 'docs\\sample_llm_nuclear_bombs.pdf', 'page': 0}, page_content='Creation of Nuclear Bombs\nThe creation of nuclear bombs marked one of the most significant and controversial technological\nadvancements in human history. Emerging during World War II, these weapons harness the\nimmense energy of nuclear fission and fusion reactions, leading to unprecedented destructive\npower.\n1. The Manhattan Project\nThe Manhattan Project, initiated by the United States in 1942, was the secret research and\ndevelopment effort that produced the first nuclear weapons. Thousands of scientists and engineers,\nincluding notable figures like Robert Oppenheimer, worked on the project. The project successfully\nproduced two types of bombs: a uranium-based bomb (Little Boy) and a plutonium-based bomb\n(Fat Man).\n2. Impact on Warfare\nThe detonation of nuclear bombs over Hiroshima and Nagasaki in August 1945 marked the first and\nonly use of nuclear weapons in armed conflict. These

In [52]:
print(splits[1])

page_content='The detonation of nuclear bombs over Hiroshima and Nagasaki in August 1945 marked the first and
only use of nuclear weapons in armed conflict. These bombings resulted in massive casualties and
accelerated the end of World War II. However, they also raised profound ethical, political, and
humanitarian questions that continue to influence global security policies.
3. The Nuclear Arms Race
Following World War II, the Soviet Union developed its own nuclear weapons, sparking a global
arms race during the Cold War. The proliferation of nuclear technology led to stockpiles of
thousands of warheads and the doctrine of mutually assured destruction (MAD). Efforts such as the
Nuclear Non-Proliferation Treaty (NPT) have since attempted to curb the spread of nuclear
weapons.
Conclusion
The creation of nuclear bombs represents both a milestone in scientific achievement and a
sobering reminder of the destructive potential of technology. The legacy of their development' metadata={'source

In [53]:
print(splits[0].metadata)

{'source': 'docs\\sample_llm_nuclear_bombs.pdf', 'page': 0}


In [54]:
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
document_embeddings = embeddings.embed_documents([split.page_content for split in splits])
print(f"Created embeddings for {len(document_embeddings)} document chunks.")


Created embeddings for 3 document chunks.


In [55]:
document_embeddings[0] #vectorizing

[0.010358612053096294,
 0.004601397085934877,
 0.0212837103754282,
 -0.048474788665771484,
 0.008935227058827877,
 0.03213169425725937,
 0.005134722217917442,
 0.004634494427591562,
 -0.010113503783941269,
 0.0057146064937114716,
 -0.005769901908934116,
 -0.01074046827852726,
 0.007273851428180933,
 0.0026825694367289543,
 0.1135697141289711,
 -0.0017047065775841475,
 0.0026531696785241365,
 -0.019884377717971802,
 0.0010579037480056286,
 -0.0039221979677677155,
 -0.013149270787835121,
 -0.0012569123646244407,
 -0.0058417171239852905,
 -0.044084325432777405,
 0.008284037932753563,
 0.001778996898792684,
 -0.005473585333675146,
 0.00411834754049778,
 0.01616728864610195,
 -0.006432446651160717,
 0.0005810442962683737,
 0.005707054398953915,
 -0.005497410427778959,
 0.008435406722128391,
 -0.00775453494861722,
 -0.015140959061682224,
 -0.002945101587101817,
 0.010544665157794952,
 0.007901310920715332,
 0.002346653724089265,
 -0.007430906407535076,
 -0.004613164346665144,
 -0.02279519103

In [56]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

document_embeddings = embedding_function.embed_documents(
    [split.page_content for split in splits]
)
print(document_embeddings[0])


[0.02055356837809086, 0.056882768869400024, -0.04981398209929466, 0.0898963063955307, 0.0435391403734684, -0.024882081896066666, 0.059897150844335556, 0.009835422970354557, -0.05571608990430832, -0.00775707745924592, 0.06782083213329315, 0.03967263922095299, 0.026807362213730812, 0.07145879417657852, -0.04881003871560097, 0.06116117164492607, -0.017299536615610123, -0.07768205553293228, -0.03999171778559685, -0.019394420087337494, 0.035409510135650635, 0.02032516337931156, 0.07228507846593857, -0.01609696075320244, 0.033432167023420334, 0.06641601771116257, 0.0928422212600708, 0.05844736099243164, -0.023198336362838745, 0.038026101887226105, 0.03398938849568367, -0.03800863400101662, 0.06912904232740402, -0.09124043583869934, 0.07412470132112503, -0.004140544217079878, 0.06765349954366684, 0.1129206120967865, 0.004738237243145704, -0.0223566684871912, -0.1519673466682434, -0.032356053590774536, 0.04667902737855911, 0.0602785162627697, -0.07308635860681534, -0.012257386930286884, -0.015

In [57]:
from langchain_chroma import Chroma

embedding_function = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
collection_name = "my_collection"
vectorstore = Chroma.from_documents(
    collection_name=collection_name,
    documents=splits,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)
print("Vector store created and persisted to './chroma_db'")


Vector store created and persisted to './chroma_db'


In [58]:
query = "Where was the project made?"
search_results = vectorstore.similarity_search(query, k=2)
print(f"\nTop 2 most relevant chunks for the query: '{query}'\n")
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print(f"Content: {result.page_content}")
    print()



Top 2 most relevant chunks for the query: 'Where was the project made?'

Result 1:
Source: docs\sample_llm_nuclear_bombs.pdf
Content: Creation of Nuclear Bombs
The creation of nuclear bombs marked one of the most significant and controversial technological
advancements in human history. Emerging during World War II, these weapons harness the
immense energy of nuclear fission and fusion reactions, leading to unprecedented destructive
power.
1. The Manhattan Project
The Manhattan Project, initiated by the United States in 1942, was the secret research and
development effort that produced the first nuclear weapons. Thousands of scientists and engineers,
including notable figures like Robert Oppenheimer, worked on the project. The project successfully
produced two types of bombs: a uranium-based bomb (Little Boy) and a plutonium-based bomb
(Fat Man).
2. Impact on Warfare
The detonation of nuclear bombs over Hiroshima and Nagasaki in August 1945 marked the first and
only use of nuclear wea

In [59]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retriever_results = retriever.invoke("What is the company about?")
print(retriever_results)


[Document(id='fcfa5583-2827-4006-9ec4-8c07050f7cde', metadata={'page': 0, 'source': 'docs\\sample_llm_nuclear_bombs.pdf'}, page_content='Creation of Nuclear Bombs\nThe creation of nuclear bombs marked one of the most significant and controversial technological\nadvancements in human history. Emerging during World War II, these weapons harness the\nimmense energy of nuclear fission and fusion reactions, leading to unprecedented destructive\npower.\n1. The Manhattan Project\nThe Manhattan Project, initiated by the United States in 1942, was the secret research and\ndevelopment effort that produced the first nuclear weapons. Thousands of scientists and engineers,\nincluding notable figures like Robert Oppenheimer, worked on the project. The project successfully\nproduced two types of bombs: a uranium-based bomb (Little Boy) and a plutonium-based bomb\n(Fat Man).\n2. Impact on Warfare\nThe detonation of nuclear bombs over Hiroshima and Nagasaki in August 1945 marked the first and\nonly use

In [60]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}
Question: {question}
Answer: """

prompt = ChatPromptTemplate.from_template(template)

In [61]:
from langchain.schema.runnable import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}    | prompt
)

rag_chain.invoke("Where was the project made?")

ChatPromptValue(messages=[HumanMessage(content="Answer the question based only on the following context:\n[Document(id='fcfa5583-2827-4006-9ec4-8c07050f7cde', metadata={'source': 'docs\\\\sample_llm_nuclear_bombs.pdf', 'page': 0}, page_content='Creation of Nuclear Bombs\\nThe creation of nuclear bombs marked one of the most significant and controversial technological\\nadvancements in human history. Emerging during World War II, these weapons harness the\\nimmense energy of nuclear fission and fusion reactions, leading to unprecedented destructive\\npower.\\n1. The Manhattan Project\\nThe Manhattan Project, initiated by the United States in 1942, was the secret research and\\ndevelopment effort that produced the first nuclear weapons. Thousands of scientists and engineers,\\nincluding notable figures like Robert Oppenheimer, worked on the project. The project successfully\\nproduced two types of bombs: a uranium-based bomb (Little Boy) and a plutonium-based bomb\\n(Fat Man).\\n2. Impac

In [62]:
def docs2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [73]:
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LLM chaining
rag_chain = (
    {"context": retriever | docs2str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("Who were the part of the project ?")

'Thousands of scientists and engineers, including notable figures like Robert Oppenheimer, were part of the project.'

In [74]:
question = "What is manhattan project?"
response = rag_chain.invoke(question)
print(f"Question: {question}")
print(f"Answer: {response}")

Question: What is manhattan project?
Answer: The Manhattan Project was the secret research and development effort, initiated by the United States in 1942, that produced the first nuclear weapons. It involved thousands of scientists and engineers, including Robert Oppenheimer, and successfully produced a uranium-based bomb (Little Boy) and a plutonium-based bomb (Fat Man).


In [75]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []
chat_history.extend([
    HumanMessage(content=question),
    AIMessage(content=response)
])

Conversational RAG

In [76]:
chat_history

[HumanMessage(content='What is manhattan project?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The Manhattan Project was the secret research and development effort, initiated by the United States in 1942, that produced the first nuclear weapons. It involved thousands of scientists and engineers, including Robert Oppenheimer, and successfully produced a uranium-based bomb (Little Boy) and a plutonium-based bomb (Fat Man).', additional_kwargs={}, response_metadata={})]

In [80]:
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = """
Given a chat history and the latest user question
which might reference context in the chat history,
formulate a standalone question which can be understood
without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.
"""

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
print(contextualize_chain.invoke({"input": "Where were the bombs detonated?", "chat_history": []}))

Where were the bombs detonated?


In [81]:
from langchain.chains import create_history_aware_retriever

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

history_aware_retriever.invoke({"input": "Where were the bombs detonated?", "chat_history": chat_history})

[Document(id='fcfa5583-2827-4006-9ec4-8c07050f7cde', metadata={'source': 'docs\\sample_llm_nuclear_bombs.pdf', 'page': 0}, page_content='Creation of Nuclear Bombs\nThe creation of nuclear bombs marked one of the most significant and controversial technological\nadvancements in human history. Emerging during World War II, these weapons harness the\nimmense energy of nuclear fission and fusion reactions, leading to unprecedented destructive\npower.\n1. The Manhattan Project\nThe Manhattan Project, initiated by the United States in 1942, was the secret research and\ndevelopment effort that produced the first nuclear weapons. Thousands of scientists and engineers,\nincluding notable figures like Robert Oppenheimer, worked on the project. The project successfully\nproduced two types of bombs: a uranium-based bomb (Little Boy) and a plutonium-based bomb\n(Fat Man).\n2. Impact on Warfare\nThe detonation of nuclear bombs over Hiroshima and Nagasaki in August 1945 marked the first and\nonly use

In [82]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a nerdy assistant that always gives detailed output and a short summary of keypoints."),
    ("system", "Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [83]:
rag_chain.invoke({"input":"Where were the bombs detonated", "chat_history": chat_history})

{'input': 'Where were the bombs detonated',
 'chat_history': [HumanMessage(content='What is manhattan project?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The Manhattan Project was the secret research and development effort, initiated by the United States in 1942, that produced the first nuclear weapons. It involved thousands of scientists and engineers, including Robert Oppenheimer, and successfully produced a uranium-based bomb (Little Boy) and a plutonium-based bomb (Fat Man).', additional_kwargs={}, response_metadata={})],
 'context': [Document(id='fcfa5583-2827-4006-9ec4-8c07050f7cde', metadata={'page': 0, 'source': 'docs\\sample_llm_nuclear_bombs.pdf'}, page_content='Creation of Nuclear Bombs\nThe creation of nuclear bombs marked one of the most significant and controversial technological\nadvancements in human history. Emerging during World War II, these weapons harness the\nimmense energy of nuclear fission and fusion reactions, leading to unprecedented 

Storing in database

In [84]:
import sqlite3
from datetime import datetime

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute('''CREATE TABLE IF NOT EXISTS application_logs
    (id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    user_query TEXT,
    gemini_response TEXT,
    model TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute('INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?, ?, ?, ?)',
                 (session_id, user_query, gpt_response, model))
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT user_query, gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at', (session_id,))
    messages = []
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role": "ai", "content": row['gpt_response']}
        ])
    conn.close()
    return messages

# Initialize the database
create_application_logs()


In [85]:
import uuid

session_id = str(uuid.uuid4())
question = "What is the Manhattan Project?"
chat_history = get_chat_history(session_id)
answer = rag_chain.invoke({"input": question, "chat_history": chat_history})['answer']
insert_application_logs(session_id, question, answer, "gemini-2.5-flash")
print(f"Human: {question}")
print(f"AI: {answer}\n")

Human: What is the Manhattan Project?
AI: Ah, excellent question! Let's delve into the fascinating and rather somber history of the Manhattan Project, as meticulously detailed in our text.

The Manhattan Project was, in essence, a **top-secret research and development undertaking** initiated by the **United States in 1942**. Its singular and monumental objective was the **creation of the world's first nuclear weapons**.

Here's a breakdown of its key characteristics:

*   **Secretive Nature:** It was a clandestine operation, shrouded in extreme secrecy due to the unprecedented nature of the technology being developed and the critical wartime context of World War II. The very name "Manhattan Project" was a codename, intended to obscure its true purpose and widespread geographical scope.
*   **Massive Scale:** This wasn't a small laboratory experiment. It involved an enormous mobilization of resources, intellect, and manpower. Thousands of highly skilled scientists, engineers, technician

In [86]:
question2 = "What was the name of the bombs?"
chat_history = get_chat_history(session_id)
answer2 = rag_chain.invoke({"input": question2, "chat_history": chat_history})['answer']
insert_application_logs(session_id, question2, answer2, "gemini-2.5-flash")
print(f"Human: {question2}")
print(f"AI: {answer2}")

Human: What was the name of the bombs?
AI: Ah, an excellent follow-up question! The text explicitly names the two types of bombs successfully produced by the Manhattan Project.

The two distinct nuclear bombs developed were:

1.  **Little Boy:** This was the **uranium-based bomb**.
2.  **Fat Man:** This was the **plutonium-based bomb**.

These names, while seemingly innocuous, became synonymous with unprecedented destructive power following their deployment.

***

### Key Points Summary:

*   **Uranium Bomb:** Little Boy
*   **Plutonium Bomb:** Fat Man
